In [3]:
# =============================================================================
# 💵 DOLAR TRACK PRO v3.0
# Sistema Profesional de Gestión y Análisis de TRM Semanal
# Arquitectura: POO + SQLite + CRUD + JOIN + Validaciones + Menú Decorado
# =============================================================================

import sqlite3
from abc import ABC, abstractmethod
from datetime import date, datetime, timedelta
from typing import List, Optional


DB_NAME = "dolar_track_pro_grupo_3.db"


# =============================================================================
# EXCEPCIONES PERSONALIZADAS
# =============================================================================

class AppError(Exception):
    """Excepción base de la aplicación."""
    pass


class ValidationError(AppError):
    """Se lanza cuando un dato no cumple las reglas de validación."""
    pass


class DatabaseError(AppError):
    """Se lanza cuando ocurre un error en la base de datos."""
    pass


class PermissionDeniedError(AppError):
    """Se lanza cuando un usuario no tiene permisos suficientes."""
    pass


class NotFoundError(AppError):
    """Se lanza cuando no se encuentra un registro solicitado."""
    pass


# =============================================================================
# CAPA 1: CLASES PADRE (ABSTRACCIÓN)
# =============================================================================

class EntidadBase(ABC):
    """
    Clase padre abstracta para entidades persistibles.
    Toda entidad puede tener ID y fecha de creación.
    """

    def __init__(self, id_entidad: Optional[int] = None, fecha_creacion: Optional[str] = None):
        self.__id_entidad = id_entidad
        self.__fecha_creacion = fecha_creacion or str(date.today())

    def get_id(self) -> Optional[int]:
        return self.__id_entidad

    def set_id(self, id_entidad: int) -> None:
        if not isinstance(id_entidad, int) or id_entidad <= 0:
            raise ValidationError("El ID debe ser un entero positivo.")
        self.__id_entidad = id_entidad

    def get_fecha_creacion(self) -> str:
        return self.__fecha_creacion

    @abstractmethod
    def to_dict(self) -> dict:
        pass


class DivisaBase(ABC):
    """
    Clase padre abstracta para representar una divisa o instrumento monetario.
    """

    def __init__(self, nombre: str, simbolo: str, descripcion: str = ""):
        self.__nombre = None
        self.__simbolo = None
        self.__descripcion = None

        self.set_nombre(nombre)
        self.set_simbolo(simbolo)
        self.set_descripcion(descripcion)

    def get_nombre(self) -> str:
        return self.__nombre

    def set_nombre(self, nombre: str) -> None:
        if not isinstance(nombre, str) or not nombre.strip():
            raise ValidationError("El nombre no puede estar vacío.")
        self.__nombre = nombre.strip().title()

    def get_simbolo(self) -> str:
        return self.__simbolo

    def set_simbolo(self, simbolo: str) -> None:
        if not isinstance(simbolo, str) or not simbolo.strip():
            raise ValidationError("El símbolo no puede estar vacío.")
        self.__simbolo = simbolo.strip().upper()

    def get_descripcion(self) -> str:
        return self.__descripcion

    def set_descripcion(self, descripcion: str) -> None:
        self.__descripcion = descripcion.strip() if isinstance(descripcion, str) else ""

    @abstractmethod
    def to_dict(self) -> dict:
        pass


class UsuarioBase(ABC):
    """
    Clase padre abstracta para los usuarios del sistema.
    """

    ROLES_VALIDOS = {"analista", "administrador", "visualizador"}

    def __init__(self, nombre: str, email: str, rol: str):
        self.__nombre = None
        self.__email = None
        self.__rol = None

        self.set_nombre(nombre)
        self.set_email(email)
        self.set_rol(rol)

    def get_nombre(self) -> str:
        return self.__nombre

    def set_nombre(self, nombre: str) -> None:
        if not isinstance(nombre, str) or not nombre.strip():
            raise ValidationError("El nombre del usuario no puede estar vacío.")
        self.__nombre = nombre.strip().title()

    def get_email(self) -> str:
        return self.__email

    def set_email(self, email: str) -> None:
        if not isinstance(email, str) or "@" not in email or "." not in email:
            raise ValidationError("El correo electrónico no tiene un formato válido.")
        self.__email = email.strip().lower()

    def get_rol(self) -> str:
        return self.__rol

    def set_rol(self, rol: str) -> None:
        rol_normalizado = rol.strip().lower()
        if rol_normalizado not in self.ROLES_VALIDOS:
            raise ValidationError(f"Rol inválido. Opciones válidas: {self.ROLES_VALIDOS}")
        self.__rol = rol_normalizado

    @abstractmethod
    def puede_eliminar(self) -> bool:
        pass

    @abstractmethod
    def to_dict(self) -> dict:
        pass


# =============================================================================
# CAPA 2: CLASES HIJO (ESPECIALIZACIÓN)
# =============================================================================

class Moneda(DivisaBase):
    """
    Clase hija de DivisaBase.
    Representa una moneda del sistema.
    """

    def to_dict(self) -> dict:
        return {
            "nombre": self.get_nombre(),
            "simbolo": self.get_simbolo(),
            "descripcion": self.get_descripcion()
        }


class RegistroTRM(EntidadBase):
    """
    Clase hija de EntidadBase.
    Representa un registro diario de TRM asociado a una moneda y a un usuario.
    """

    def __init__(
        self,
        id_moneda: int,
        id_usuario: int,
        fecha: str,
        valor: float,
        id_entidad: Optional[int] = None,
        fecha_creacion: Optional[str] = None
    ):
        super().__init__(id_entidad, fecha_creacion)

        self.__id_moneda = None
        self.__id_usuario = None
        self.__fecha = None
        self.__valor = None

        self.set_id_moneda(id_moneda)
        self.set_id_usuario(id_usuario)
        self.set_fecha(fecha)
        self.set_valor(valor)

    def get_id_moneda(self) -> int:
        return self.__id_moneda

    def set_id_moneda(self, id_moneda: int) -> None:
        if not isinstance(id_moneda, int) or id_moneda <= 0:
            raise ValidationError("El ID de la moneda debe ser un entero positivo.")
        self.__id_moneda = id_moneda

    def get_id_usuario(self) -> int:
        return self.__id_usuario

    def set_id_usuario(self, id_usuario: int) -> None:
        if not isinstance(id_usuario, int) or id_usuario <= 0:
            raise ValidationError("El ID del usuario debe ser un entero positivo.")
        self.__id_usuario = id_usuario

    def get_fecha(self) -> str:
        return self.__fecha

    def set_fecha(self, fecha: str) -> None:
        try:
            datetime.strptime(fecha, "%Y-%m-%d")
        except ValueError:
            raise ValidationError("La fecha debe tener el formato YYYY-MM-DD.")
        self.__fecha = fecha

    def get_valor(self) -> float:
        return self.__valor

    def set_valor(self, valor: float) -> None:
        try:
            valor = float(valor)
        except (ValueError, TypeError):
            raise ValidationError("El valor de la TRM debe ser numérico.")

        if valor < 500 or valor > 15000:
            raise ValidationError("El valor de la TRM está fuera del rango permitido.")
        self.__valor = round(valor, 2)

    def to_dict(self) -> dict:
        return {
            "id_moneda": self.get_id_moneda(),
            "id_usuario": self.get_id_usuario(),
            "fecha": self.get_fecha(),
            "valor": self.get_valor()
        }


class AnalisisSemanal(EntidadBase):
    """
    Clase hija de EntidadBase.
    Representa un análisis semanal construido a partir de 7 registros de TRM.
    """

    def __init__(
        self,
        id_usuario: int,
        fecha_inicio: str,
        fecha_fin: str,
        promedio: float,
        volatilidad: float,
        dias_venta: int,
        dias_compra: int,
        id_entidad: Optional[int] = None,
        fecha_creacion: Optional[str] = None
    ):
        super().__init__(id_entidad, fecha_creacion)

        self.__id_usuario = id_usuario
        self.__fecha_inicio = fecha_inicio
        self.__fecha_fin = fecha_fin
        self.__promedio = round(float(promedio), 2)
        self.__volatilidad = round(float(volatilidad), 2)
        self.__dias_venta = int(dias_venta)
        self.__dias_compra = int(dias_compra)

    def get_id_usuario(self) -> int:
        return self.__id_usuario

    def get_fecha_inicio(self) -> str:
        return self.__fecha_inicio

    def get_fecha_fin(self) -> str:
        return self.__fecha_fin

    def get_promedio(self) -> float:
        return self.__promedio

    def get_volatilidad(self) -> float:
        return self.__volatilidad

    def get_dias_venta(self) -> int:
        return self.__dias_venta

    def get_dias_compra(self) -> int:
        return self.__dias_compra

    def resumen(self) -> str:
        return (
            f"Promedio semanal           : ${self.__promedio:,.2f}\n"
            f"Volatilidad semanal        : ${self.__volatilidad:,.2f}\n"
            f"Días recomendados de venta : {self.__dias_venta}\n"
            f"Días recomendados de compra: {self.__dias_compra}"
        )

    def to_dict(self) -> dict:
        return {
            "id_usuario": self.__id_usuario,
            "fecha_inicio": self.__fecha_inicio,
            "fecha_fin": self.__fecha_fin,
            "promedio": self.__promedio,
            "volatilidad": self.__volatilidad,
            "dias_venta": self.__dias_venta,
            "dias_compra": self.__dias_compra
        }


class Analista(UsuarioBase):
    """
    Clase hija de UsuarioBase.
    Puede consultar, crear y actualizar, pero no eliminar.
    """
    def __init__(self, nombre: str, email: str):
        super().__init__(nombre, email, "analista")

    def puede_eliminar(self) -> bool:
        return False

    def to_dict(self) -> dict:
        return {
            "nombre": self.get_nombre(),
            "email": self.get_email(),
            "rol": self.get_rol()
        }


class Administrador(UsuarioBase):
    """
    Clase hija de UsuarioBase.
    Tiene permisos completos.
    """
    def __init__(self, nombre: str, email: str):
        super().__init__(nombre, email, "administrador")

    def puede_eliminar(self) -> bool:
        return True

    def to_dict(self) -> dict:
        return {
            "nombre": self.get_nombre(),
            "email": self.get_email(),
            "rol": self.get_rol()
        }


class Visualizador(UsuarioBase):
    """
    Clase hija de UsuarioBase.
    Solo puede consultar información.
    """
    def __init__(self, nombre: str, email: str):
        super().__init__(nombre, email, "visualizador")

    def puede_eliminar(self) -> bool:
        return False

    def to_dict(self) -> dict:
        return {
            "nombre": self.get_nombre(),
            "email": self.get_email(),
            "rol": self.get_rol()
        }


# =============================================================================
# CAPA 3: INFRAESTRUCTURA DE BASE DE DATOS
# =============================================================================

class DatabaseManager:
    """
    Gestiona la conexión, ejecución de consultas y transacciones SQLite.
    """

    def __init__(self, db_name: str = DB_NAME):
        self.__db_name = db_name
        self.__conn: Optional[sqlite3.Connection] = None

    def connect(self) -> sqlite3.Connection:
        if self.__conn is None:
            try:
                self.__conn = sqlite3.connect(self.__db_name)
                self.__conn.row_factory = sqlite3.Row
                self.__conn.execute("PRAGMA foreign_keys = ON;")
            except sqlite3.Error as e:
                raise DatabaseError(f"No fue posible conectar a la base de datos: {e}")
        return self.__conn

    def close(self) -> None:
        if self.__conn:
            self.__conn.close()
            self.__conn = None

    def execute(self, query: str, params: tuple = ()) -> sqlite3.Cursor:
        try:
            conn = self.connect()
            cursor = conn.cursor()
            cursor.execute(query, params)
            conn.commit()
            return cursor
        except sqlite3.Error as e:
            conn.rollback()
            raise DatabaseError(f"Error en la operación SQL: {e}")

    def executemany(self, query: str, seq_of_params: list) -> sqlite3.Cursor:
        try:
            conn = self.connect()
            cursor = conn.cursor()
            cursor.executemany(query, seq_of_params)
            conn.commit()
            return cursor
        except sqlite3.Error as e:
            conn.rollback()
            raise DatabaseError(f"Error en operación masiva SQL: {e}")

    def fetchall(self, query: str, params: tuple = ()) -> List[sqlite3.Row]:
        try:
            conn = self.connect()
            cursor = conn.cursor()
            cursor.execute(query, params)
            return cursor.fetchall()
        except sqlite3.Error as e:
            raise DatabaseError(f"Error consultando datos: {e}")

    def fetchone(self, query: str, params: tuple = ()) -> Optional[sqlite3.Row]:
        try:
            conn = self.connect()
            cursor = conn.cursor()
            cursor.execute(query, params)
            return cursor.fetchone()
        except sqlite3.Error as e:
            raise DatabaseError(f"Error consultando dato: {e}")


# =============================================================================
# CAPA 4: INICIALIZADOR DE BASE DE DATOS
# =============================================================================

class DatabaseInitializer:
    """
    Crea las tablas y siembra datos iniciales si no existen.
    """

    def __init__(self, db: DatabaseManager):
        self.__db = db

    def create_tables(self) -> None:
        self.__db.execute("""
            CREATE TABLE IF NOT EXISTS monedas (
                id_moneda    INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre       TEXT NOT NULL,
                simbolo      TEXT NOT NULL UNIQUE,
                descripcion  TEXT
            );
        """)

        self.__db.execute("""
            CREATE TABLE IF NOT EXISTS usuarios (
                id_usuario   INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre       TEXT NOT NULL,
                email        TEXT NOT NULL UNIQUE,
                rol          TEXT NOT NULL CHECK(rol IN ('analista', 'administrador', 'visualizador'))
            );
        """)

        self.__db.execute("""
            CREATE TABLE IF NOT EXISTS registros_trm (
                id_registro   INTEGER PRIMARY KEY AUTOINCREMENT,
                id_moneda     INTEGER NOT NULL,
                id_usuario    INTEGER NOT NULL,
                fecha         TEXT NOT NULL,
                valor         REAL NOT NULL CHECK(valor > 0),
                UNIQUE(id_moneda, fecha),
                FOREIGN KEY (id_moneda) REFERENCES monedas(id_moneda) ON UPDATE CASCADE ON DELETE RESTRICT,
                FOREIGN KEY (id_usuario) REFERENCES usuarios(id_usuario) ON UPDATE CASCADE ON DELETE RESTRICT
            );
        """)

        self.__db.execute("""
            CREATE TABLE IF NOT EXISTS analisis_semanal (
                id_analisis    INTEGER PRIMARY KEY AUTOINCREMENT,
                id_usuario     INTEGER NOT NULL,
                fecha_inicio   TEXT NOT NULL,
                fecha_fin      TEXT NOT NULL,
                promedio       REAL NOT NULL,
                volatilidad    REAL NOT NULL,
                dias_venta     INTEGER NOT NULL,
                dias_compra    INTEGER NOT NULL,
                fecha_calculo  TEXT NOT NULL,
                FOREIGN KEY (id_usuario) REFERENCES usuarios(id_usuario) ON UPDATE CASCADE ON DELETE RESTRICT
            );
        """)

    def seed_data(self) -> None:
        row = self.__db.fetchone("SELECT COUNT(*) AS total FROM monedas;")
        if row["total"] == 0:
            monedas = [
                ("Dólar Estadounidense", "USD/COP", "Tasa Representativa del Mercado"),
                ("Euro", "EUR/COP", "Moneda de la Unión Europea"),
                ("Libra Esterlina", "GBP/COP", "Moneda del Reino Unido"),
                ("Yen Japonés", "JPY/COP", "Moneda de Japón"),
                ("Franco Suizo", "CHF/COP", "Moneda de Suiza"),
            ]
            self.__db.executemany(
                "INSERT INTO monedas (nombre, simbolo, descripcion) VALUES (?, ?, ?);",
                monedas
            )

        row = self.__db.fetchone("SELECT COUNT(*) AS total FROM usuarios;")
        if row["total"] == 0:
            usuarios = [
                ("Carlos Administrador", "carlos@dolartrack.co", "administrador"),
                ("Laura Analista", "laura@dolartrack.co", "analista"),
                ("Pedro Analista", "pedro@dolartrack.co", "analista"),
                ("Diana Visualizadora", "diana@dolartrack.co", "visualizador"),
                ("Mateo Analista", "mateo@dolartrack.co", "analista"),
            ]
            self.__db.executemany(
                "INSERT INTO usuarios (nombre, email, rol) VALUES (?, ?, ?);",
                usuarios
            )

        row = self.__db.fetchone("SELECT COUNT(*) AS total FROM registros_trm;")
        if row["total"] == 0:
            hoy = date.today()
            valores = [4198.50, 4210.75, 4185.00, 4225.30, 4240.00, 4195.80, 4230.60]
            registros = []
            for i, valor in enumerate(valores):
                fecha = str(hoy - timedelta(days=6 - i))
                registros.append((1, 1, fecha, valor))

            self.__db.executemany(
                "INSERT INTO registros_trm (id_moneda, id_usuario, fecha, valor) VALUES (?, ?, ?, ?);",
                registros
            )

        row = self.__db.fetchone("SELECT COUNT(*) AS total FROM analisis_semanal;")
        if row["total"] == 0:
            hoy = date.today()
            analisis = [
                (1, str(hoy - timedelta(days=13)), str(hoy - timedelta(days=7)), 4120.50, 85.00, 3, 4, str(hoy - timedelta(days=7))),
                (2, str(hoy - timedelta(days=6)), str(hoy), 4212.14, 55.00, 4, 3, str(hoy)),
                (3, str(hoy - timedelta(days=20)), str(hoy - timedelta(days=14)), 4095.90, 72.15, 2, 5, str(hoy - timedelta(days=14))),
            ]
            self.__db.executemany("""
                INSERT INTO analisis_semanal
                (id_usuario, fecha_inicio, fecha_fin, promedio, volatilidad, dias_venta, dias_compra, fecha_calculo)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?);
            """, analisis)


# =============================================================================
# CAPA 5: REPOSITORIOS
# =============================================================================

class UsuarioRepository:
    """
    Repositorio para la tabla usuarios.
    """

    def __init__(self, db: DatabaseManager):
        self.__db = db

    def create(self, usuario: UsuarioBase) -> int:
        cursor = self.__db.execute(
            "INSERT INTO usuarios (nombre, email, rol) VALUES (?, ?, ?);",
            (usuario.get_nombre(), usuario.get_email(), usuario.get_rol())
        )
        return cursor.lastrowid

    def get_all(self) -> List[sqlite3.Row]:
        return self.__db.fetchall(
            "SELECT id_usuario, nombre, email, rol FROM usuarios ORDER BY id_usuario;"
        )

    def get_by_id(self, id_usuario: int) -> sqlite3.Row:
        row = self.__db.fetchone(
            "SELECT id_usuario, nombre, email, rol FROM usuarios WHERE id_usuario = ?;",
            (id_usuario,)
        )
        if not row:
            raise NotFoundError(f"No existe un usuario con ID {id_usuario}.")
        return row

    def update_role(self, id_usuario: int, nuevo_rol: str) -> None:
        cursor = self.__db.execute(
            "UPDATE usuarios SET rol = ? WHERE id_usuario = ?;",
            (nuevo_rol, id_usuario)
        )
        if cursor.rowcount == 0:
            raise NotFoundError(f"No existe un usuario con ID {id_usuario}.")

    def delete(self, id_usuario: int) -> None:
        cursor = self.__db.execute(
            "DELETE FROM usuarios WHERE id_usuario = ?;",
            (id_usuario,)
        )
        if cursor.rowcount == 0:
            raise NotFoundError(f"No existe un usuario con ID {id_usuario}.")


class MonedaRepository:
    """
    Repositorio para la tabla monedas.
    """

    def __init__(self, db: DatabaseManager):
        self.__db = db

    def get_all(self) -> List[sqlite3.Row]:
        return self.__db.fetchall(
            "SELECT id_moneda, nombre, simbolo, descripcion FROM monedas ORDER BY id_moneda;"
        )

    def get_by_id(self, id_moneda: int) -> sqlite3.Row:
        row = self.__db.fetchone(
            "SELECT id_moneda, nombre, simbolo, descripcion FROM monedas WHERE id_moneda = ?;",
            (id_moneda,)
        )
        if not row:
            raise NotFoundError(f"No existe una moneda con ID {id_moneda}.")
        return row


class RegistroTRMRepository:
    """
    Repositorio para la tabla registros_trm.
    """

    def __init__(self, db: DatabaseManager):
        self.__db = db

    def create(self, registro: RegistroTRM) -> int:
        cursor = self.__db.execute("""
            INSERT INTO registros_trm (id_moneda, id_usuario, fecha, valor)
            VALUES (?, ?, ?, ?);
        """, (
            registro.get_id_moneda(),
            registro.get_id_usuario(),
            registro.get_fecha(),
            registro.get_valor()
        ))
        return cursor.lastrowid

    def get_all_with_join(self) -> List[sqlite3.Row]:
        return self.__db.fetchall("""
            SELECT
                r.id_registro,
                r.fecha,
                r.valor,
                m.simbolo,
                m.nombre AS moneda,
                u.nombre AS usuario,
                u.rol
            FROM registros_trm r
            INNER JOIN monedas m ON r.id_moneda = m.id_moneda
            INNER JOIN usuarios u ON r.id_usuario = u.id_usuario
            ORDER BY r.fecha DESC, r.id_registro DESC;
        """)

    def get_last_7_by_moneda(self, id_moneda: int) -> List[sqlite3.Row]:
        return self.__db.fetchall("""
            SELECT id_registro, fecha, valor
            FROM registros_trm
            WHERE id_moneda = ?
            ORDER BY fecha DESC
            LIMIT 7;
        """, (id_moneda,))

    def update_value(self, id_registro: int, nuevo_valor: float) -> None:
        cursor = self.__db.execute(
            "UPDATE registros_trm SET valor = ? WHERE id_registro = ?;",
            (round(nuevo_valor, 2), id_registro)
        )
        if cursor.rowcount == 0:
            raise NotFoundError(f"No existe un registro con ID {id_registro}.")

    def delete(self, id_registro: int) -> None:
        cursor = self.__db.execute(
            "DELETE FROM registros_trm WHERE id_registro = ?;",
            (id_registro,)
        )
        if cursor.rowcount == 0:
            raise NotFoundError(f"No existe un registro con ID {id_registro}.")


class AnalisisRepository:
    """
    Repositorio para la tabla analisis_semanal.
    """

    def __init__(self, db: DatabaseManager):
        self.__db = db

    def create(self, analisis: AnalisisSemanal) -> int:
        cursor = self.__db.execute("""
            INSERT INTO analisis_semanal
            (id_usuario, fecha_inicio, fecha_fin, promedio, volatilidad, dias_venta, dias_compra, fecha_calculo)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?);
        """, (
            analisis.get_id_usuario(),
            analisis.get_fecha_inicio(),
            analisis.get_fecha_fin(),
            analisis.get_promedio(),
            analisis.get_volatilidad(),
            analisis.get_dias_venta(),
            analisis.get_dias_compra(),
            str(date.today())
        ))
        return cursor.lastrowid

    def get_all(self) -> List[sqlite3.Row]:
        return self.__db.fetchall("""
            SELECT a.id_analisis, u.nombre AS usuario, a.fecha_inicio, a.fecha_fin,
                   a.promedio, a.volatilidad, a.dias_venta, a.dias_compra, a.fecha_calculo
            FROM analisis_semanal a
            INNER JOIN usuarios u ON a.id_usuario = u.id_usuario
            ORDER BY a.id_analisis DESC;
        """)


# =============================================================================
# CAPA 6: SERVICIOS
# =============================================================================

class UsuarioService:
    """
    Lógica de negocio para usuarios.
    """

    def __init__(self, repo: UsuarioRepository):
        self.__repo = repo

    def listar(self) -> List[sqlite3.Row]:
        return self.__repo.get_all()

    def obtener(self, id_usuario: int) -> sqlite3.Row:
        return self.__repo.get_by_id(id_usuario)

    def crear(self, nombre: str, email: str, rol: str) -> int:
        rol = rol.strip().lower()

        if rol == "administrador":
            usuario = Administrador(nombre, email)
        elif rol == "analista":
            usuario = Analista(nombre, email)
        elif rol == "visualizador":
            usuario = Visualizador(nombre, email)
        else:
            raise ValidationError("Rol inválido. Debe ser analista, administrador o visualizador.")

        return self.__repo.create(usuario)

    def actualizar_rol(self, id_usuario: int, nuevo_rol: str, usuario_activo: UsuarioBase) -> None:
        if not usuario_activo.puede_eliminar():
            raise PermissionDeniedError("Solo un administrador puede cambiar roles.")

        rol = nuevo_rol.strip().lower()
        if rol not in UsuarioBase.ROLES_VALIDOS:
            raise ValidationError("Rol inválido.")

        self.__repo.update_role(id_usuario, rol)

    def eliminar(self, id_usuario: int, usuario_activo: UsuarioBase) -> None:
        if not usuario_activo.puede_eliminar():
            raise PermissionDeniedError("Solo un administrador puede eliminar usuarios.")
        self.__repo.delete(id_usuario)


class RegistroTRMService:
    """
    Lógica de negocio para registros de TRM.
    """

    def __init__(
        self,
        repo: RegistroTRMRepository,
        moneda_repo: MonedaRepository,
        usuario_repo: UsuarioRepository
    ):
        self.__repo = repo
        self.__moneda_repo = moneda_repo
        self.__usuario_repo = usuario_repo

    def registrar_semana(self, id_moneda: int, id_usuario: int, valores: List[float], fecha_inicio: date) -> List[int]:
        if len(valores) != 7:
            raise ValidationError("Se deben ingresar exactamente 7 valores.")

        self.__moneda_repo.get_by_id(id_moneda)
        self.__usuario_repo.get_by_id(id_usuario)

        ids = []
        for i, valor in enumerate(valores):
            fecha = str(fecha_inicio + timedelta(days=i))
            registro = RegistroTRM(id_moneda, id_usuario, fecha, valor)
            ids.append(self.__repo.create(registro))
        return ids

    def listar_todos_con_join(self) -> List[sqlite3.Row]:
        return self.__repo.get_all_with_join()

    def actualizar_valor(self, id_registro: int, nuevo_valor: float, usuario_activo: UsuarioBase) -> None:
        if usuario_activo.get_rol() not in {"analista", "administrador"}:
            raise PermissionDeniedError("No tiene permisos para actualizar registros.")
        RegistroTRM(1, 1, str(date.today()), nuevo_valor)
        self.__repo.update_value(id_registro, nuevo_valor)

    def eliminar_registro(self, id_registro: int, usuario_activo: UsuarioBase) -> None:
        if not usuario_activo.puede_eliminar():
            raise PermissionDeniedError("Solo un administrador puede eliminar registros.")
        self.__repo.delete(id_registro)

    def obtener_ultimos_7(self, id_moneda: int) -> List[sqlite3.Row]:
        registros = self.__repo.get_last_7_by_moneda(id_moneda)
        if len(registros) < 7:
            raise ValidationError("No hay suficientes registros para calcular el análisis semanal.")
        return registros


class AnalisisService:
    """
    Lógica de negocio para análisis semanales.
    """

    def __init__(self, repo: AnalisisRepository):
        self.__repo = repo

    def calcular_analisis(self, id_usuario: int, registros: List[sqlite3.Row]) -> AnalisisSemanal:
        if len(registros) != 7:
            raise ValidationError("El análisis requiere exactamente 7 registros.")

        valores = [float(r["valor"]) for r in registros]
        fechas = [r["fecha"] for r in registros]

        promedio = sum(valores) / len(valores)
        volatilidad = max(valores) - min(valores)
        dias_venta = sum(1 for v in valores if v > promedio)
        dias_compra = sum(1 for v in valores if v < promedio)

        return AnalisisSemanal(
            id_usuario=id_usuario,
            fecha_inicio=min(fechas),
            fecha_fin=max(fechas),
            promedio=promedio,
            volatilidad=volatilidad,
            dias_venta=dias_venta,
            dias_compra=dias_compra
        )

    def guardar(self, analisis: AnalisisSemanal) -> int:
        return self.__repo.create(analisis)

    def listar(self) -> List[sqlite3.Row]:
        return self.__repo.get_all()


# =============================================================================
# CAPA 7: UTILIDADES DE ENTRADA
# =============================================================================

class InputHelper:
    """
    Ayuda a leer entradas desde consola con validación.
    """

    @staticmethod
    def leer_int(mensaje: str, minimo: int = 1, maximo: int = 999999) -> int:
        while True:
            try:
                valor = int(input(mensaje).strip())
                if minimo <= valor <= maximo:
                    return valor
                print(f"⚠️ Debe ingresar un número entre {minimo} y {maximo}.")
            except ValueError:
                print("⚠️ Entrada inválida. Debe ingresar un número entero.")

    @staticmethod
    def leer_float(mensaje: str) -> float:
        while True:
            try:
                return float(input(mensaje).strip())
            except ValueError:
                print("⚠️ Entrada inválida. Debe ingresar un número decimal válido.")

    @staticmethod
    def leer_str(mensaje: str) -> str:
        while True:
            valor = input(mensaje).strip()
            if valor:
                return valor
            print("⚠️ Este campo no puede estar vacío.")


# =============================================================================
# CAPA 8: INTERFAZ / APLICACIÓN
# =============================================================================

class DolarTrackApp:
    """
    Clase principal que ejecuta el menú y coordina toda la aplicación.
    """

    SEP = "═" * 84
    SUB = "─" * 84

    def __init__(self):
        self.__db = DatabaseManager()
        self.__initializer = DatabaseInitializer(self.__db)

        self.__usuario_repo = UsuarioRepository(self.__db)
        self.__moneda_repo = MonedaRepository(self.__db)
        self.__registro_repo = RegistroTRMRepository(self.__db)
        self.__analisis_repo = AnalisisRepository(self.__db)

        self.__usuario_service = UsuarioService(self.__usuario_repo)
        self.__registro_service = RegistroTRMService(
            self.__registro_repo, self.__moneda_repo, self.__usuario_repo
        )
        self.__analisis_service = AnalisisService(self.__analisis_repo)

        self.__usuario_activo: Optional[UsuarioBase] = None
        self.__usuario_activo_id: Optional[int] = None

    # -------------------------------------------------------------------------
    # UTILIDADES VISUALES
    # -------------------------------------------------------------------------
    def _print_banner(self) -> None:
        print("\n" + self.SEP)
        print("💵📈 DOLAR TRACK PRO v3.0 | Sistema Profesional de Gestión y Análisis TRM")
        print("🏛️ Arquitectura POO + SQLite + CRUD + JOIN + Validaciones + Menú Interactivo")
        print(self.SEP)

    def _print_header(self, titulo: str, emoji: str = "📌") -> None:
        print("\n" + self.SEP)
        print(f"{emoji}  {titulo.upper()}")
        print(self.SEP)

    def _print_section(self, titulo: str) -> None:
        print(f"\n🔹 {titulo}")
        print(self.SUB)

    def _print_success(self, mensaje: str) -> None:
        print(f"\n✅ {mensaje}")

    def _print_error(self, mensaje: str) -> None:
        print(f"\n❌ {mensaje}")

    def _print_info(self, mensaje: str) -> None:
        print(f"\nℹ️ {mensaje}")

    def _print_warning(self, mensaje: str) -> None:
        print(f"\n⚠️ {mensaje}")

    def _pause(self) -> None:
        input("\n⏳ Presione ENTER para continuar...")

    def _crear_objeto_usuario_desde_fila(self, fila: sqlite3.Row) -> UsuarioBase:
        rol = fila["rol"]
        nombre = fila["nombre"]
        email = fila["email"]

        if rol == "administrador":
            return Administrador(nombre, email)
        elif rol == "analista":
            return Analista(nombre, email)
        else:
            return Visualizador(nombre, email)

    # -------------------------------------------------------------------------
    # LOGIN
    # -------------------------------------------------------------------------
    def iniciar_sesion(self) -> None:
        self._print_header("Inicio de Sesión", "🔐")
        usuarios = self.__usuario_service.listar()

        print("👥 Usuarios disponibles:\n")
        for u in usuarios:
            print(f"   [{u['id_usuario']}] 👤 {u['nombre']} | 📧 {u['email']} | 🛡️ {u['rol']}")

        id_usuario = InputHelper.leer_int("\n👉 Ingrese el ID del usuario: ")
        fila = self.__usuario_service.obtener(id_usuario)

        self.__usuario_activo = self._crear_objeto_usuario_desde_fila(fila)
        self.__usuario_activo_id = fila["id_usuario"]

        self._print_success(
            f"Bienvenido/a {self.__usuario_activo.get_nombre()} "
            f"({self.__usuario_activo.get_rol()})"
        )

    # -------------------------------------------------------------------------
    # MENÚ PRINCIPAL
    # -------------------------------------------------------------------------
    def menu_principal(self) -> None:
        while True:
            self._print_header("Menú Principal", "🏠")
            print(f"👤 Usuario activo : {self.__usuario_activo.get_nombre()}")
            print(f"🛡️ Rol actual     : {self.__usuario_activo.get_rol()}")
            print(self.SUB)
            print("1️⃣  Gestión de registros TRM")
            print("2️⃣  Calcular análisis semanal")
            print("3️⃣  Gestión de usuarios")
            print("4️⃣  Ver análisis guardados")
            print("5️⃣  Cambiar usuario")
            print("0️⃣  Salir")

            opcion = input("\n👉 Seleccione una opción: ").strip()

            try:
                if opcion == "1":
                    self.menu_registros()
                elif opcion == "2":
                    self.menu_analisis()
                elif opcion == "3":
                    self.menu_usuarios()
                elif opcion == "4":
                    self.ver_analisis_guardados()
                elif opcion == "5":
                    self.iniciar_sesion()
                elif opcion == "0":
                    print("\n👋 Gracias por usar Dolar Track Pro. ¡Hasta pronto!")
                    break
                else:
                    self._print_warning("Opción no válida.")
            except AppError as e:
                self._print_error(str(e))
                self._pause()

    # -------------------------------------------------------------------------
    # MENÚ REGISTROS
    # -------------------------------------------------------------------------
    def menu_registros(self) -> None:
        while True:
            self._print_header("Gestión de Registros TRM", "💹")
            print("1️⃣  Registrar TRM de los últimos 7 días")
            print("2️⃣  Listar registros con JOIN")
            print("3️⃣  Actualizar valor de un registro")
            print("4️⃣  Eliminar un registro")
            print("0️⃣  Volver")

            opcion = input("\n👉 Seleccione una opción: ").strip()

            try:
                if opcion == "1":
                    self.registrar_semana()
                elif opcion == "2":
                    self.listar_registros()
                elif opcion == "3":
                    self.actualizar_registro()
                elif opcion == "4":
                    self.eliminar_registro()
                elif opcion == "0":
                    break
                else:
                    self._print_warning("Opción no válida.")
            except AppError as e:
                self._print_error(str(e))
                self._pause()

    def registrar_semana(self) -> None:
        self._print_header("Registrar Semana TRM", "📝")

        monedas = self.__moneda_repo.get_all()
        self._print_section("Monedas disponibles")
        for m in monedas:
            print(f"[{m['id_moneda']}] 💱 {m['nombre']} ({m['simbolo']})")

        id_moneda = InputHelper.leer_int("\n👉 Seleccione el ID de la moneda: ")

        valores = []
        fecha_inicio = date.today() - timedelta(days=6)

        self._print_section("Ingreso de valores diarios")
        for i in range(7):
            fecha_actual = fecha_inicio + timedelta(days=i)
            valor = InputHelper.leer_float(f"📆 Ingrese la TRM del día {fecha_actual}: ")
            valores.append(valor)

        ids = self.__registro_service.registrar_semana(
            id_moneda=id_moneda,
            id_usuario=self.__usuario_activo_id,
            valores=valores,
            fecha_inicio=fecha_inicio
        )

        self._print_success(f"Se insertaron correctamente {len(ids)} registros.")
        self._pause()

    def listar_registros(self) -> None:
        self._print_header("Listado de Registros con JOIN", "📋")
        registros = self.__registro_service.listar_todos_con_join()

        if not registros:
            self._print_info("No hay registros almacenados.")
        else:
            print(f"{'ID':<5}{'Fecha':<12}{'Valor':<14}{'Símbolo':<12}{'Moneda':<25}{'Usuario':<22}{'Rol'}")
            print(self.SUB)
            for r in registros:
                print(
                    f"{r['id_registro']:<5}"
                    f"{r['fecha']:<12}"
                    f"${r['valor']:<13,.2f}"
                    f"{r['simbolo']:<12}"
                    f"{r['moneda']:<25}"
                    f"{r['usuario']:<22}"
                    f"{r['rol']}"
                )
        self._pause()

    def actualizar_registro(self) -> None:
        self._print_header("Actualizar Registro", "✏️")
        id_registro = InputHelper.leer_int("👉 Ingrese el ID del registro a actualizar: ")
        nuevo_valor = InputHelper.leer_float("💲 Ingrese el nuevo valor TRM: ")

        self.__registro_service.actualizar_valor(
            id_registro=id_registro,
            nuevo_valor=nuevo_valor,
            usuario_activo=self.__usuario_activo
        )

        self._print_success("Registro actualizado correctamente.")
        self._pause()

    def eliminar_registro(self) -> None:
        self._print_header("Eliminar Registro", "🗑️")
        id_registro = InputHelper.leer_int("👉 Ingrese el ID del registro a eliminar: ")
        confirmacion = input("❓ ¿Está seguro? (s/n): ").strip().lower()

        if confirmacion == "s":
            self.__registro_service.eliminar_registro(
                id_registro=id_registro,
                usuario_activo=self.__usuario_activo
            )
            self._print_success("Registro eliminado correctamente.")
        else:
            self._print_info("Operación cancelada.")
        self._pause()

    # -------------------------------------------------------------------------
    # MENÚ ANÁLISIS
    # -------------------------------------------------------------------------
    def menu_analisis(self) -> None:
        self._print_header("Análisis Semanal", "📊")

        monedas = self.__moneda_repo.get_all()
        self._print_section("Monedas disponibles")
        for m in monedas:
            print(f"[{m['id_moneda']}] 💱 {m['nombre']} ({m['simbolo']})")

        id_moneda = InputHelper.leer_int("\n👉 Seleccione el ID de la moneda a analizar: ")
        registros = self.__registro_service.obtener_ultimos_7(id_moneda)

        analisis = self.__analisis_service.calcular_analisis(
            id_usuario=self.__usuario_activo_id,
            registros=registros
        )

        self._print_section("Resultado del análisis")
        print(f"📅 Fecha inicial : {analisis.get_fecha_inicio()}")
        print(f"📅 Fecha final   : {analisis.get_fecha_fin()}")
        print(f"💰 Promedio      : ${analisis.get_promedio():,.2f}")
        print(f"📈 Volatilidad   : ${analisis.get_volatilidad():,.2f}")
        print(f"🟢 Días venta    : {analisis.get_dias_venta()}")
        print(f"🔵 Días compra   : {analisis.get_dias_compra()}")

        guardar = input("\n💾 ¿Desea guardar este análisis? (s/n): ").strip().lower()
        if guardar == "s":
            analisis_id = self.__analisis_service.guardar(analisis)
            self._print_success(f"Análisis guardado con ID {analisis_id}.")
        else:
            self._print_info("El análisis no fue guardado.")

        self._pause()

    def ver_analisis_guardados(self) -> None:
        self._print_header("Análisis Guardados", "🧠")
        analisis = self.__analisis_service.listar()

        if not analisis:
            self._print_info("No hay análisis guardados.")
        else:
            print(f"{'ID':<5}{'Usuario':<22}{'Inicio':<12}{'Fin':<12}{'Promedio':<14}{'Volatilidad':<14}")
            print(self.SUB)
            for a in analisis:
                print(
                    f"{a['id_analisis']:<5}"
                    f"{a['usuario']:<22}"
                    f"{a['fecha_inicio']:<12}"
                    f"{a['fecha_fin']:<12}"
                    f"${a['promedio']:<13,.2f}"
                    f"${a['volatilidad']:<13,.2f}"
                )
        self._pause()

    # -------------------------------------------------------------------------
    # MENÚ USUARIOS
    # -------------------------------------------------------------------------
    def menu_usuarios(self) -> None:
        while True:
            self._print_header("Gestión de Usuarios", "👥")
            print("1️⃣  Listar usuarios")
            print("2️⃣  Crear usuario")
            print("3️⃣  Cambiar rol de usuario")
            print("4️⃣  Eliminar usuario")
            print("0️⃣  Volver")

            opcion = input("\n👉 Seleccione una opción: ").strip()

            try:
                if opcion == "1":
                    self.listar_usuarios()
                elif opcion == "2":
                    self.crear_usuario()
                elif opcion == "3":
                    self.cambiar_rol_usuario()
                elif opcion == "4":
                    self.eliminar_usuario()
                elif opcion == "0":
                    break
                else:
                    self._print_warning("Opción no válida.")
            except AppError as e:
                self._print_error(str(e))
                self._pause()

    def listar_usuarios(self) -> None:
        self._print_header("Usuarios del Sistema", "📇")
        usuarios = self.__usuario_service.listar()

        print(f"{'ID':<5}{'Nombre':<28}{'Email':<32}{'Rol'}")
        print(self.SUB)
        for u in usuarios:
            print(f"{u['id_usuario']:<5}{u['nombre']:<28}{u['email']:<32}{u['rol']}")
        self._pause()

    def crear_usuario(self) -> None:
        self._print_header("Crear Usuario", "➕")
        nombre = InputHelper.leer_str("👤 Nombre completo: ")
        email = InputHelper.leer_str("📧 Email: ")
        rol = InputHelper.leer_str("🛡️ Rol (analista/administrador/visualizador): ")

        nuevo_id = self.__usuario_service.crear(nombre, email, rol)
        self._print_success(f"Usuario creado correctamente con ID {nuevo_id}.")
        self._pause()

    def cambiar_rol_usuario(self) -> None:
        self._print_header("Cambiar Rol de Usuario", "🔄")
        id_usuario = InputHelper.leer_int("👉 Ingrese el ID del usuario: ")
        nuevo_rol = InputHelper.leer_str("🛡️ Nuevo rol: ")

        self.__usuario_service.actualizar_rol(
            id_usuario=id_usuario,
            nuevo_rol=nuevo_rol,
            usuario_activo=self.__usuario_activo
        )
        self._print_success("Rol actualizado correctamente.")
        self._pause()

    def eliminar_usuario(self) -> None:
        self._print_header("Eliminar Usuario", "🚫")
        id_usuario = InputHelper.leer_int("👉 Ingrese el ID del usuario a eliminar: ")
        confirmacion = input("❓ ¿Confirma la eliminación? (s/n): ").strip().lower()

        if confirmacion == "s":
            self.__usuario_service.eliminar(
                id_usuario=id_usuario,
                usuario_activo=self.__usuario_activo
            )
            self._print_success("Usuario eliminado correctamente.")
        else:
            self._print_info("Operación cancelada.")
        self._pause()

    # -------------------------------------------------------------------------
    # EJECUCIÓN
    # -------------------------------------------------------------------------
    def run(self) -> None:
        try:
            self._print_banner()
            self._print_info("Inicializando base de datos y verificando estructura...")
            self.__db.connect()
            self.__initializer.create_tables()
            self.__initializer.seed_data()
            self._print_success("Base de datos lista y datos semilla verificados.")

            self.iniciar_sesion()
            self.menu_principal()

        except AppError as e:
            self._print_error(f"Error de aplicación: {e}")
        except Exception as e:
            self._print_error(f"Error inesperado: {e}")
        finally:
            self.__db.close()
            print("\n🔒 Conexión cerrada correctamente.")


# =============================================================================
# PUNTO DE ENTRADA
# =============================================================================

if __name__ == "__main__":
    app = DolarTrackApp()
    app.run()


════════════════════════════════════════════════════════════════════════════════════
💵📈 DOLAR TRACK PRO v3.0 | Sistema Profesional de Gestión y Análisis TRM
🏛️ Arquitectura POO + SQLite + CRUD + JOIN + Validaciones + Menú Interactivo
════════════════════════════════════════════════════════════════════════════════════

ℹ️ Inicializando base de datos y verificando estructura...

✅ Base de datos lista y datos semilla verificados.

════════════════════════════════════════════════════════════════════════════════════
🔐  INICIO DE SESIÓN
════════════════════════════════════════════════════════════════════════════════════
👥 Usuarios disponibles:

   [1] 👤 Carlos Administrador | 📧 carlos@dolartrack.co | 🛡️ administrador
   [2] 👤 Laura Analista | 📧 laura@dolartrack.co | 🛡️ analista
   [3] 👤 Pedro Analista | 📧 pedro@dolartrack.co | 🛡️ analista
   [4] 👤 Diana Visualizadora | 📧 diana@dolartrack.co | 🛡️ visualizador
   [5] 👤 Mateo Analista | 📧 mateo@dolartrack.co | 🛡️ analista

✅ Bienvenido/a Diana

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("dolar_track_pro_grupo_3.db")

df_usuarios = pd.read_sql_query("SELECT * FROM usuarios;", conn)
df_monedas = pd.read_sql_query("SELECT * FROM monedas;", conn)
df_registros = pd.read_sql_query("SELECT * FROM registros_trm;", conn)
df_analisis = pd.read_sql_query("SELECT * FROM analisis_semanal;", conn)

print("Usuarios")
display(df_usuarios)

print("Monedas")
display(df_monedas)

print("Registros TRM")
display(df_registros)

print("Análisis semanal")
display(df_analisis)

conn.close()

Usuarios


,id_usuario,nombre,email,rol
0,1,Carlos Administrador,carlos@dolartrack.co,administrador
1,2,Laura Analista,laura@dolartrack.co,analista
2,3,Pedro Analista,pedro@dolartrack.co,analista
3,4,Diana Visualizadora,diana@dolartrack.co,visualizador
4,5,Mateo Analista,mateo@dolartrack.co,analista


Monedas


,id_moneda,nombre,simbolo,descripcion
0,1,Dólar Estadounidense,USD/COP,Tasa Representativa del Mercado
1,2,Euro,EUR/COP,Moneda de la Unión Europea
2,3,Libra Esterlina,GBP/COP,Moneda del Reino Unido
3,4,Yen Japonés,JPY/COP,Moneda de Japón
4,5,Franco Suizo,CHF/COP,Moneda de Suiza


Registros TRM


,id_registro,id_moneda,id_usuario,fecha,valor
0,1,1,1,2026-04-01,4198.50
1,2,1,1,2026-04-02,4210.75
2,3,1,1,2026-04-03,4185.00
3,4,1,1,2026-04-04,4225.30
4,5,1,1,2026-04-05,4240.00
5,6,1,1,2026-04-06,4195.80
6,7,1,1,2026-04-07,4230.60


Análisis semanal


,id_analisis,id_usuario,fecha_inicio,fecha_fin,promedio,volatilidad,dias_venta,dias_compra,fecha_calculo
0,1,1,2026-03-25,2026-03-31,4120.50,85.00,3,4,2026-03-31
1,2,2,2026-04-01,2026-04-07,4212.14,55.00,4,3,2026-04-07
2,3,3,2026-03-18,2026-03-24,4095.90,72.15,2,5,2026-03-24
